# FunnyBirds MCBM Recall Gap Analysis — γ sweep

Parallel to `funnybirds_cbm_recall.ipynb` (FunnyBirds baseline/CBM) but extended to
**Minimal Concept Bottleneck Models (MCBM)** (Almudévar et al., arXiv:2506.04877).

MCBM adds an Information Bottleneck (IB) penalty that forces each concept representation
z_j to be a *minimal* sufficient statistic for concept c_j, suppressing species-identity
leakage (backwash).

**Central question:** As γ (IB penalty strength) increases, does the recall gap decrease?
γ=0.0 recovers standard CBM (sanity check).

**FunnyBirds advantages over CUB:**
- Exact class-concept matrix (no annotation noise)
- 26 concepts, each all-positive or all-negative per species
- 50 species, 5 body parts (beak/wing/tail/foot/eye)
- Concept overlap between species is exactly computable

γ values evaluated: configurable via `GAMMAS` below.
Feature dirs: `features/resnet50_mcbm_funnybirds_gamma{γ}/`

For each attribute:

1. Train a linear probe on top of frozen visual features.
2. Evaluate the probe on held-out test images.
3. Group test images by species.
4. For pairs of all-positive species (FunnyBirds: attributes are fixed per species):
   - Sample images and compute recall for each species.
5. Measure the recall gap between species.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

## 0. Configuration

- `GAMMAS` — IB penalty strengths to sweep. γ=0.0 = standard CBM (sanity check).
- `LAYERS` — 18 ResNet-50 probe points for emergence analysis.
- `EMERGE_METRIC` — `"balanced"` (balanced accuracy) or `"plain"` (accuracy) for emergence curves.
- `EMERGE_CRITERION` — `"frac90"` (first layer reaching 90% of final) or `"sharp_jump"` (largest step).
- `EMERGE_FRAC` — fraction threshold for frac90 criterion (default 0.90).

In [ ]:
LAYERS = [
    "conv1",
    "layer1.0", "layer1.1", "layer1.2",
    "layer2.0", "layer2.1", "layer2.2", "layer2.3",
    "layer3.0", "layer3.1", "layer3.2", "layer3.3", "layer3.4", "layer3.5",
    "layer4.0", "layer4.1", "layer4.2",
    "avgpool",
]

EMERGE_METRIC    = "balanced"    # "balanced" | "plain"
EMERGE_FRAC      = 0.90          # threshold for frac90 criterion
EMERGE_CRITERION = "sharp_jump"  # "frac90" | "sharp_jump"
emerge_col = "emerge_idx_frac"   if EMERGE_CRITERION == "frac90" else "emerge_idx_jump"
layer_col  = "emerge_layer_frac" if EMERGE_CRITERION == "frac90" else "emerge_layer_jump"

GAMMAS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]

# Analysis layer for matched-pair recall gap (species-identity emergence layer)
LAYER            = "layer4.0"
SPECIES_LAYER_IDX = 14  # index of layer4.0 in LAYERS

print(f"GAMMAS: {GAMMAS}")
print(f"LAYER:  {LAYER}")
print(f"EMERGE_METRIC={EMERGE_METRIC}  EMERGE_CRITERION={EMERGE_CRITERION}  EMERGE_FRAC={EMERGE_FRAC}")

## Paths

- `FB` — FunnyBirds dataset root
- `MCBM_FB_FEATS[γ]` — pre-computed ResNet-50 MCBM features for FunnyBirds at IB strength γ

Features are extracted by `run_fb_mcbm_extract_adroit.sh`. If a gamma's features aren't
ready yet, it's excluded from `GAMMAS` with a warning (no assertion error).

In [ ]:
import sys
ROOT = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB             = ROOT / 'data' / 'FunnyBirds'
MCBM_FB_FEATS  = {g: ROOT / 'features' / f'resnet50_mcbm_funnybirds_gamma{g}' for g in GAMMAS}

def _feats_ready(feat_dir: Path) -> bool:
    """Check that labels files (not just the directory) exist."""
    return (
        (feat_dir / 'labels_train.pt').exists() and
        (feat_dir / 'labels_test.pt').exists()
    )

_missing = [g for g in GAMMAS if not _feats_ready(MCBM_FB_FEATS[g])]
if _missing:
    print(f'[warn] Missing MCBM FunnyBirds features for gamma={_missing}')
    print('       Run run_fb_mcbm_extract_adroit.sh to extract them.')
    GAMMAS = [g for g in GAMMAS if _feats_ready(MCBM_FB_FEATS[g])]
    print(f'       Continuing with available gammas: {GAMMAS}')

if not GAMMAS:
    raise RuntimeError('No MCBM FunnyBirds features are available. Extract features first.')

assert FB.exists(), f'Missing FunnyBirds folder: {FB}'
assert (FB / 'dataset_train.json').exists(), f'Missing dataset_train.json: {FB}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[config] device: {device}')
print(f'[config] available gammas: {GAMMAS}')

In [ ]:
def load_species_maps(fb_root: Path):
    """
    Load species ID→name mappings from FunnyBirds metadata/classes.csv.
    Mirrors load_species_maps(cub_root) in mcbm_recall_full.ipynb.
    """
    classes_csv = fb_root / 'metadata' / 'classes.csv'
    if not classes_csv.exists():
        raise FileNotFoundError(
            f'metadata/classes.csv not found. Run prepare_funnybirds_metadata.py first.'
        )
    df = pd.read_csv(classes_csv)
    id2name  = dict(zip(df['class_id'], df['class_name']))
    id2short = {k: v.replace('funnybird_', 'FB') for k, v in id2name.items()}
    return id2name, id2short


def load_meta(fb_root: Path) -> pd.DataFrame:
    """
    Load per-image metadata with species information.
    Returns DataFrame: image_id, class_id, species_id, species_name, is_train.
    """
    images_csv = fb_root / 'metadata' / 'images.csv'
    if not images_csv.exists():
        raise FileNotFoundError(
            f'metadata/images.csv not found. Run prepare_funnybirds_metadata.py first.'
        )
    df = pd.read_csv(images_csv)
    id2name, _ = load_species_maps(fb_root)
    df['species_id']   = df['class_id']
    df['species_name'] = df['class_id'].map(id2name)
    return df


def load_image_attr_labels_robust(fb_root: Path) -> pd.DataFrame:
    """
    Load per-image binary concept labels from metadata/image_concepts_binary.csv.
    Returns long-form DataFrame: image_id, attr_id, attr_name, is_present, certainty.
    certainty is always 1 for FunnyBirds (ground-truth, no annotation noise).
    """
    concepts_csv = fb_root / 'metadata' / 'image_concepts_binary.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f'metadata/image_concepts_binary.csv not found. '
            f'Run prepare_funnybirds_metadata.py first.'
        )
    wide = pd.read_csv(concepts_csv)
    concept_cols = [c for c in wide.columns if c != 'image_id']
    long = wide.melt(id_vars='image_id', value_vars=concept_cols,
                     var_name='attr_name', value_name='is_present')
    long['attr_id']    = long.groupby('attr_name', sort=False).ngroup()
    long['is_present'] = long['is_present'].astype(int)
    long['certainty']  = 1  # FunnyBirds: ground-truth, always certain
    return long[['image_id', 'attr_id', 'attr_name', 'is_present', 'certainty']]


def load_attr_maps(fb_root: Path):
    """
    Load concept names from metadata/concepts.csv.
    Returns: attr_id_to_name, attr_name_to_id.
    """
    concepts_csv = fb_root / 'metadata' / 'concepts.csv'
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f'metadata/concepts.csv not found. Run prepare_funnybirds_metadata.py first.'
        )
    df = pd.read_csv(concepts_csv)
    id2name  = dict(zip(df['concept_id'], df['concept_name']))
    name2id  = dict(zip(df['concept_name'], df['concept_id']))
    return id2name, name2id

In [ ]:
# FunnyBirds ground-truth class-concept matrix
# Every row is a species; every column is one of 26 part-variant concepts.
# This matrix is EXACT (no annotation noise) — the key FunnyBirds advantage.

from datasets.funnybirds_dataset import FunnyBirdsDataset
from datasets.funnybirds_dataset import concept_names as _cnames

_fb_ds = FunnyBirdsDataset(FB, split='train')
class_concept_matrix, _ = _fb_ds.get_class_concept_matrix()

cc_df = pd.DataFrame(
    class_concept_matrix.numpy(),
    columns=_cnames(),
    index=[f'funnybird_{i:02d}' for i in range(class_concept_matrix.shape[0])],
)
_sid_min = 0  # FunnyBirds species IDs start at 0 (adjust if metadata differs)

print(f'Class-concept matrix shape: {cc_df.shape}  (species × concepts)')
print('Each row should sum to 5 (one variant per part):')
print(cc_df.sum(axis=1).value_counts())
cc_df.head()

In [ ]:
# Load metadata, concept labels, and attribute maps
meta           = load_meta(FB)
img_attr_long  = load_image_attr_labels_robust(FB)
attr_id_to_name, attr_name_to_id = load_attr_maps(FB)

attr_df = pd.DataFrame({
    'attr_name': list(attr_name_to_id.keys()),
    'attr_id':   list(attr_name_to_id.values()),
})

id2name, _ = load_species_maps(FB)
spname = lambda sid: id2name.get(int(sid), f'funnybird_{int(sid):02d}')

print(f'meta:          {len(meta)} images  ({meta["is_train"].sum()} train, {(meta["is_train"]==0).sum()} test)')
print(f'img_attr_long: {len(img_attr_long)} rows, {img_attr_long["attr_name"].nunique()} concepts')
print(f'attr_df:       {len(attr_df)} concepts')
print(f'Concepts:      {list(attr_name_to_id.keys())}')

## 1. Concept selection

FunnyBirds has exactly **26 binary concepts** (one-hot over part variants):

| Part  | Variants | Concept dims |
|-------|----------|--------------|
| beak  | 4        | beak_0 .. beak_3 |
| eye   | 3        | eye_0  .. eye_2  |
| wing  | 6        | wing_0 .. wing_5 |
| foot  | 4        | foot_0 .. foot_3 |
| tail  | 9        | tail_0 .. tail_8 |

Each species uses exactly one variant per part → concept vector is one-hot per part group.
No 10-90% prevalence filter needed: by construction each variant appears in 50/n_variants species.

`SCREENED_ATTR_LIST` is populated after screening in §4.

In [ ]:
# FunnyBirds: all 26 concepts loaded directly from dataset module (canonical source)
from datasets.funnybirds_dataset import concept_names as _fb_concept_names
ATTR_LIST = _fb_concept_names()
print(f'FunnyBirds concepts ({len(ATTR_LIST)}): {ATTR_LIST}')


## 2. Loading pre-computed features

`load_split_order` reads `labels_{split}.pt` to get the DataLoader image-ID ordering.

All MCBM γ values share the same FunnyBirds DataLoader order (same dataset, same seed),
so we load split order **once** from `GAMMAS[0]` and verify the rest match.

In [ ]:
def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def to_1d_int_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.array(x).reshape(-1)
    return x.astype(int)


def infer_kind(arr):
    if arr.max() <= 200 and arr.min() >= 0:
        return 'species_id_like'
    if arr.max() > 200:
        return 'image_id_like'
    return 'unknown'


def load_split_order(feat_dir: Path, split: str):
    p = feat_dir / f'labels_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    t = safe_torch_load(p)
    assert isinstance(t, dict), f'Expected dict in {p}, got {type(t)}'
    assert 'image_ids' in t, f'{p} missing image_ids key; has {list(t.keys())}'
    ids  = to_1d_int_array(t['image_ids'])
    kind = infer_kind(ids)
    return kind, ids


def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    p = feat_dir / f'{layer}_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


# Load split order from first available gamma (all MCBM FunnyBirds models share the same
# DataLoader ordering — verified by cross-gamma consistency check below)
_ref_gamma = GAMMAS[0]
_ref_feat  = MCBM_FB_FEATS[_ref_gamma]
mcbm_kind_tr, mcbm_ids_tr = load_split_order(_ref_feat, 'train')
mcbm_kind_te, mcbm_ids_te = load_split_order(_ref_feat, 'test')

# Verify all other gammas share the same DataLoader order
for g in GAMMAS[1:]:
    _, ids_tr_g = load_split_order(MCBM_FB_FEATS[g], 'train')
    _, ids_te_g = load_split_order(MCBM_FB_FEATS[g], 'test')
    assert np.array_equal(mcbm_ids_tr, ids_tr_g), f'gamma={g} train split order differs!'
    assert np.array_equal(mcbm_ids_te, ids_te_g), f'gamma={g} test split order differs!'

print(f'Split order kind: {mcbm_kind_tr}  id range: ({mcbm_ids_tr.min()}, {mcbm_ids_tr.max()})')
print(f'Train N={len(mcbm_ids_tr)}  Test N={len(mcbm_ids_te)}')
print(f'Cross-gamma consistency: OK (all {len(GAMMAS)} gammas share same order)')

In [ ]:
def align_features_and_labels(
    X_split: torch.Tensor,
    image_ids_in_feature_order: np.ndarray,
    labeled_df_split: pd.DataFrame,
):
    """
    Align feature tensor rows to attribute labels by image_id.
    Returns X_aligned [M, D] and df_aligned [M rows] in the same order.
    """
    labeled = labeled_df_split.set_index('image_id')[['y', 'species_id', 'species_name']]
    keep_idx = []
    rows = []
    for i, img_id in enumerate(image_ids_in_feature_order):
        img_id = int(img_id)
        if img_id in labeled.index:
            keep_idx.append(i)
            y, sid, sname = labeled.loc[img_id]
            rows.append((img_id, int(sid), str(sname), int(y)))
    X_aligned  = X_split[keep_idx]
    df_aligned = pd.DataFrame(rows, columns=['image_id', 'species_id', 'species_name', 'y'])
    return X_aligned, df_aligned

## 3. Linear probe — training and evaluation

For each attribute we train a **1-layer linear probe** on frozen backbone features
at layer `LAYER` (default: `layer4.0`).

| Setting | Value |
|---|---|
| Loss | `BCEWithLogitsLoss` with `pos_weight` (class-imbalance correction) |
| Optimiser | `AdamW`, lr = 1e-2, wd = 1e-4 |
| Epochs | 25 |
| Batch size | 512 |

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.lin = nn.Linear(d, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)


def train_probe(Xtr: torch.Tensor, ytr: np.ndarray,
                seed=0, lr=1e-2, wd=1e-4, epochs=25, batch=512):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    Xtr   = Xtr.to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)
    probe = LinearProbe(Xtr.shape[1]).to(device)
    opt   = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)

    pos = float(ytr_t.mean().item())
    pos_weight = (
        torch.tensor([(1 - pos) / pos], device=device)
        if 0 < pos < 1 else torch.tensor([1.0], device=device)
    )
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    n = Xtr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx    = perm[i:i+batch]
            logits = probe(Xtr[idx])
            loss   = loss_fn(logits, ytr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return probe


@torch.no_grad()
def predict_probs(probe: nn.Module, X: torch.Tensor, batch=4096) -> np.ndarray:
    probe.eval()
    probs = []
    for i in range(0, X.shape[0], batch):
        xb = X[i:i+batch].to(device)
        probs.append(torch.sigmoid(probe(xb)).detach().cpu())
    return torch.cat(probs, dim=0).numpy()


print('Defined: LinearProbe  train_probe  predict_probs')

In [ ]:
def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    return (float(np.quantile(x, alpha/2)), float(np.quantile(x, 1 - alpha/2)))


def bootstrap_p_value(values, null=0.0):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return np.nan
    return float(2.0 * min(np.mean(v <= null), np.mean(v >= null)))


def make_candidate_pairs_fb(
    df_test: pd.DataFrame, min_pos: int = 3, max_pairs: int = 200, seed: int = 0
):
    """
    FunnyBirds: pair all-positive species (prevalence >= 0.9, n_pos >= min_pos).
    Returns list of (sid_A, sid_B, mpos) 3-tuples.
    """
    from itertools import combinations as _combinations
    g   = df_test.groupby('species_id')['y'].agg(['count', 'sum']).rename(columns={'sum': 'pos'})
    g['prev'] = g['pos'] / g['count']
    ok  = g[(g['pos'] >= min_pos) & (g['prev'] >= 0.9)]
    sids = ok.index.tolist()
    pairs = []
    for a, b in _combinations(sids, 2):
        mpos = int(min(ok.loc[a, 'pos'], ok.loc[b, 'pos']))
        pairs.append((int(a), int(b), mpos))
        if len(pairs) >= max_pairs:
            break
    return pairs


def fb_pair_eval(
    df_test: pd.DataFrame, probs: np.ndarray,
    sid_A: int, sid_B: int, mpos: int,
    seed: int = 0, thr: float = 0.5,
):
    """
    Sample mpos images (with replacement) from each all-positive species.
    Gap = |recall_A - recall_B|. replace=True ensures bootstrap runs differ.
    """
    df = df_test.copy()
    df['prob'] = probs
    A = df[(df.species_id == sid_A) & (df.y == 1)]
    B = df[(df.species_id == sid_B) & (df.y == 1)]
    n_A = min(mpos, len(A))
    n_B = min(mpos, len(B))
    A_s = A.sample(n_A, random_state=seed, replace=(n_A < mpos))
    B_s = B.sample(n_B, random_state=seed, replace=(n_B < mpos))

    def recall_pos(d):
        pred = (d.prob.values >= thr).astype(int)
        return float(pred.mean()) if len(d) > 0 else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)
    return {
        'sid_A': sid_A, 'sid_B': sid_B,
        'species_A': spname(sid_A), 'species_B': spname(sid_B),
        'npos': int(mpos), 'nneg': 0,
        'recall_A': float(recA) if recA is not None else np.nan,
        'recall_B': float(recB) if recB is not None else np.nan,
        'gap': float(abs(recA - recB)) if (recA is not None and recB is not None) else np.nan,
    }


def fb_bootstrap_summary(df_te, probs, pairs, *, thr=0.5, B=300):
    """
    FunnyBirds bootstrap summary for 3-tuple pairs (sid_A, sid_B, mpos).
    Returns same column schema as matched_pair_bootstrap_summary.
    """
    SUMMARY_COLS = [
        'sid_A','sid_B','species_A','species_B',
        'npos','nneg','gap_mean','gap_std','gap_ci_lo','gap_ci_hi','gap_p',
        'gap_ci_width','gap_snr','gap_norm','n_runs',
    ]
    LONG_COLS = ['sid_A','sid_B','species_A','species_B','npos','nneg','recall_A','recall_B','gap','boot_id']
    if not pairs:
        return pd.DataFrame(columns=LONG_COLS), pd.DataFrame(columns=SUMMARY_COLS)

    rows = []
    for (a, b, mpos) in pairs:
        for boot_id in range(B):
            r = fb_pair_eval(df_te, probs, int(a), int(b), int(mpos), seed=boot_id, thr=thr)
            r['boot_id'] = boot_id
            rows.append(r)

    res_long = pd.DataFrame(rows).dropna(subset=['gap'])
    if res_long.empty:
        return res_long, pd.DataFrame(columns=SUMMARY_COLS)

    def _ci_lo(x): return bootstrap_ci(x)[0]
    def _ci_hi(x): return bootstrap_ci(x)[1]

    pair_summary = (
        res_long.groupby(['sid_A','sid_B','species_A','species_B'], as_index=False)
                .agg(
                    npos=('npos','min'), nneg=('nneg','min'),
                    gap_mean=('gap','mean'), gap_std=('gap','std'),
                    gap_ci_lo=('gap', _ci_lo), gap_ci_hi=('gap', _ci_hi),
                    gap_p=('gap', bootstrap_p_value),
                    n_runs=('gap','size'),
                )
    )
    EPS = 1e-12
    pair_summary['gap_ci_width'] = pair_summary['gap_ci_hi'] - pair_summary['gap_ci_lo']
    pair_summary['gap_snr']  = pair_summary['gap_mean'] / (pair_summary['gap_std'].fillna(0.0) + EPS)
    pair_summary['gap_norm'] = pair_summary['gap_mean']
    pair_summary = pair_summary.sort_values('gap_mean', ascending=False).reset_index(drop=True)
    return res_long, pair_summary


def species_recall_prevalence_table(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5) -> pd.DataFrame:
    df = df_te[['species_id','species_name','y']].copy()
    df['prob'] = np.asarray(probs, dtype=float)
    df['pred'] = (df['prob'] >= thr).astype(int)
    g = df.groupby(['species_id','species_name'], as_index=False).agg(
        n=('y','size'), n_pos=('y','sum'), n_pred_pos=('pred','sum'),
    )
    g['n_neg']      = g['n'] - g['n_pos']
    g['prevalence'] = g['n_pos'] / g['n']
    tp = (df[df['y']==1].groupby(['species_id','species_name'])['pred']
            .sum().reset_index(name='tp'))
    out = g.merge(tp, on=['species_id','species_name'], how='left')
    out['tp']        = out['tp'].fillna(0).astype(int)
    out['recall']    = np.where(out['n_pos'] > 0, out['tp'] / out['n_pos'], np.nan)
    out['precision'] = np.where(out['n_pred_pos'] > 0, out['tp'] / out['n_pred_pos'], np.nan)
    return out.sort_values('n', ascending=False).reset_index(drop=True)


def add_species_bootstrap_ci(df_te, probs, thr=0.5, B=300, min_pos_for_ci=1):
    df = df_te[['species_id','species_name','y']].copy()
    df['prob'] = np.asarray(probs, dtype=float)
    rows = []
    rng  = np.random.default_rng(0)
    for (sid, sname), d in df.groupby(['species_id','species_name']):
        d     = d.reset_index(drop=True)
        n     = len(d)
        n_pos = int(d['y'].sum())
        if n_pos < min_pos_for_ci:
            rows.append({'species_id': int(sid), 'species_name': str(sname),
                         'recall_bs_mean': np.nan, 'recall_ci_lo': np.nan,
                         'recall_ci_hi': np.nan, 'recall_ci_width': np.nan, 'B': B})
            continue
        vals = []
        for b in range(B):
            idx = rng.integers(0, n, size=n)
            s   = d.iloc[idx]
            pos = s[s['y'] == 1]
            if len(pos) == 0:
                vals.append(np.nan); continue
            vals.append(float((pos['prob'].to_numpy() >= thr).mean()))
        vals = np.asarray(vals, dtype=float)
        lo, hi = bootstrap_ci(vals)
        rows.append({'species_id': int(sid), 'species_name': str(sname),
                     'recall_bs_mean': float(np.nanmean(vals)),
                     'recall_ci_lo': lo, 'recall_ci_hi': hi,
                     'recall_ci_width': (hi - lo) if np.isfinite(lo) and np.isfinite(hi) else np.nan,
                     'B': B})
    ci_df = pd.DataFrame(rows)
    base  = species_recall_prevalence_table(df_te, probs, thr=thr)
    return base.merge(ci_df, on=['species_id','species_name'], how='left')


print('Defined: bootstrap_ci  bootstrap_p_value  make_candidate_pairs_fb  fb_pair_eval')
print('         fb_bootstrap_summary  species_recall_prevalence_table  add_species_bootstrap_ci')

In [ ]:
def build_attr_labeled_df(
    meta: pd.DataFrame,
    img_attr_long: pd.DataFrame,
    attr_id: int,
    min_certainty: int = 1,
) -> pd.DataFrame:
    """
    Returns: image_id, species_id, species_name, is_train, y, certainty.
    """
    sub = img_attr_long[img_attr_long['attr_id'] == int(attr_id)].copy()
    sub = sub[sub['certainty'] >= int(min_certainty)].copy()
    out = meta.merge(sub[['image_id','is_present','certainty']], on='image_id', how='inner')
    out = out.rename(columns={'is_present': 'y'})
    out['y'] = out['y'].astype(int)
    return out[['image_id','species_id','species_name','is_train','y','certainty']]


# Sanity check
_attr = 'beak_0'
_aid  = attr_name_to_id[_attr]
_lab  = build_attr_labeled_df(meta, img_attr_long, _aid)
print(f'Attribute: {_attr}  rows: {len(_lab)}  pos_rate: {_lab.y.mean():.3f}')

In [ ]:
def run_one_attribute(
    attr_name: str,
    feat_dir: Path,
    split_order_kind_train: str,
    split_order_train: np.ndarray,
    split_order_kind_test: str,
    split_order_test: np.ndarray,
    layer: str,
    *,
    Xtr_all: torch.Tensor = None,
    Xte_all: torch.Tensor = None,
    min_certainty: int = 1,
    thr: float = 0.5,
    epochs: int = 25,
    min_each: int = 10,
    n_pairs: int = 200,
    B_gap: int = 100,
    B_species: int = 100,
    use_balanced_acc: bool = None,
):
    """
    Full pipeline for one attribute on FunnyBirds MCBM features.
    Xtr_all / Xte_all can be pre-loaded tensors (load once per gamma, reuse across attrs).
    use_balanced_acc defaults to (EMERGE_METRIC == 'balanced') if None.
    Always uses fb_bootstrap_summary (FunnyBirds has no within-species negatives).
    """
    if use_balanced_acc is None:
        use_balanced_acc = (EMERGE_METRIC == 'balanced')

    assert split_order_kind_train == 'image_id_like' and split_order_kind_test == 'image_id_like', (
        'labels_{split}.pt must be image_id indexed for alignment.'
    )

    aid      = attr_name_to_id[attr_name]
    lab      = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty)
    lab_train = lab[lab['is_train'] == 1].copy()
    lab_test  = lab[lab['is_train'] == 0].copy()

    if Xtr_all is None:
        Xtr_all = load_features(feat_dir, layer, 'train')
    if Xte_all is None:
        Xte_all = load_features(feat_dir, layer, 'test')

    Xtr, df_tr = align_features_and_labels(Xtr_all, split_order_train, lab_train)
    Xte, df_te = align_features_and_labels(Xte_all, split_order_test,  lab_test)

    ytr = df_tr['y'].astype(int).to_numpy()
    yte = df_te['y'].astype(int).to_numpy()

    probe = train_probe(Xtr, ytr, seed=0, epochs=epochs)
    probs = predict_probs(probe, Xte)

    species_table = (
        add_species_bootstrap_ci(df_te, probs, thr=thr, B=B_species)
        if B_species > 0
        else species_recall_prevalence_table(df_te, probs, thr=thr)
    )

    if len(yte) == 0:
        test_acc = np.nan
    elif use_balanced_acc:
        pred_bin = (probs >= thr).astype(int)
        tp = int(((pred_bin == 1) & (yte == 1)).sum())
        tn = int(((pred_bin == 0) & (yte == 0)).sum())
        fp = int(((pred_bin == 1) & (yte == 0)).sum())
        fn = int(((pred_bin == 0) & (yte == 1)).sum())
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        test_acc = 0.5 * (tpr + tnr)
    else:
        test_acc = float(((probs >= thr).astype(int) == yte).mean())

    if n_pairs is None or int(n_pairs) <= 0:
        res_long = pd.DataFrame()
        pair_summary = pd.DataFrame()
        pairs = []
    else:
        # FunnyBirds: always use fb_bootstrap_summary (no within-species negatives)
        pairs = make_candidate_pairs_fb(
            df_te, min_pos=max(1, min_each // 5), max_pairs=n_pairs
        )
        res_long, pair_summary = fb_bootstrap_summary(
            df_te, probs, pairs, thr=thr, B=B_gap
        )

    mean_gap = float(pair_summary['gap_mean'].mean()) if len(pair_summary) > 0 else np.nan
    p90_gap  = float(pair_summary['gap_mean'].quantile(0.9)) if len(pair_summary) > 0 else np.nan

    info = {
        'attr': attr_name, 'layer': layer,
        'n_train': int(len(df_tr)), 'n_test': int(len(df_te)),
        'train_pos_rate': float(ytr.mean()) if len(ytr) else np.nan,
        'test_pos_rate': float(yte.mean()) if len(yte) else np.nan,
        'test_acc': float(test_acc), 'thr': float(thr), 'epochs': int(epochs),
        'n_pairs': int(len(pairs)), 'B_gap': int(B_gap), 'B_species': int(B_species),
        'mean_gap': mean_gap, 'p90_gap': p90_gap,
        'use_balanced_acc': bool(use_balanced_acc),
    }
    return info, res_long, pair_summary, df_te, species_table


print('Defined: run_one_attribute')

## 4. Attribute screening

Tests all 26 FunnyBirds concepts and selects those with meaningful cross-species recall
variation.

**FunnyBirds-tuned thresholds:**
- `min_pos_per_species=3` — each positive species has all test images positive (≥10 test/class)
- `min_species_with_pos=4` — tail has ~5-6 species per variant; 4 is safe floor
- `min_overall_prev=0.01, max_overall_prev=0.99` — tail variants cover ~10% prevalence

Uses `GAMMAS[0]` features for screening (most information-rich, least IB compression).

In [ ]:
def screen_attributes_for_species_variation(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    min_pos_per_species: int = 3,
    min_species_with_pos: int = 4,
    min_overall_prev: float = 0.01,
    max_overall_prev: float = 0.99,
    epochs: int = 8,
    max_attrs=None,
    verbose_every: int = 5,
    B_species: int = 100,
):
    rows, errors = [], []
    stats = dict(tried=0, success=0,
                 filtered_too_few_species_pos=0,
                 filtered_prev_out_of_range=0,
                 filtered_no_recall_vals=0,
                 errored=0)

    cand = list(candidate_attrs)
    if max_attrs is not None:
        cand = cand[:max_attrs]

    for i, attr in enumerate(cand):
        stats['tried'] += 1
        try:
            info, _, _, _, species_table = run_one_attribute(
                attr, feat_dir, kind_tr, ids_tr, kind_te, ids_te,
                layer=layer, min_certainty=min_certainty, thr=thr,
                epochs=epochs, n_pairs=0, B_species=B_species,
            )
            st          = species_table.copy()
            overall_prev = float(st['n_pos'].sum() / st['n'].sum()) if st['n'].sum() > 0 else np.nan
            st_pos      = st[st['n_pos'] >= min_pos_per_species]
            n_sp_pos    = int(len(st_pos))

            if n_sp_pos < min_species_with_pos:
                stats['filtered_too_few_species_pos'] += 1; continue
            if not (min_overall_prev <= overall_prev <= max_overall_prev):
                stats['filtered_prev_out_of_range'] += 1; continue

            recall_vals = st_pos['recall'].dropna().to_numpy()
            if recall_vals.size == 0:
                stats['filtered_no_recall_vals'] += 1; continue

            stats['success'] += 1
            rows.append({
                'attr': attr, 'overall_prev': overall_prev, 'n_species_pos': n_sp_pos,
                'recall_std': float(np.std(recall_vals)),
                'recall_range': float(np.max(recall_vals) - np.min(recall_vals)),
                'recall_p90_p10': float(np.quantile(recall_vals, 0.9) - np.quantile(recall_vals, 0.1)),
                'test_acc': float(info['test_acc']), 'n_test': int(info['n_test']),
            })

            if verbose_every and (i + 1) % verbose_every == 0:
                print(f'[{i+1}/{len(cand)}] {attr}  prev={overall_prev:.3f}  n_sp_pos={n_sp_pos}')

        except Exception as e:
            stats['errored'] += 1
            if len(errors) < 5:
                errors.append((attr, repr(e)))

    screen_df = pd.DataFrame(rows)
    print('\n--- Screening summary ---')
    for k, v in stats.items(): print(f'  {k}: {v}')
    if errors:
        print('\nExample errors:')
        for a, msg in errors: print(f'  {a} -> {msg}')
    if screen_df.empty:
        return screen_df
    return screen_df.sort_values(
        ['recall_p90_p10','recall_range','recall_std'], ascending=False
    ).reset_index(drop=True)


# ── Run screening on GAMMAS[0] features ──────────────────────────────────────
screen_df = screen_attributes_for_species_variation(
    ATTR_LIST,
    feat_dir=MCBM_FB_FEATS[GAMMAS[0]],
    kind_tr=mcbm_kind_tr, ids_tr=mcbm_ids_tr,
    kind_te=mcbm_kind_te, ids_te=mcbm_ids_te,
    layer=LAYER,
    min_pos_per_species=3, min_species_with_pos=4,
    min_overall_prev=0.01, max_overall_prev=0.99,
    epochs=8, verbose_every=5,
)
print(f'\nScreened: {len(screen_df)} / {len(ATTR_LIST)} concepts passed')


if screen_df.empty:
    print('[warn] No concepts passed — retrying with looser thresholds')
    screen_df = screen_attributes_for_species_variation(
        ATTR_LIST,
        feat_dir=MCBM_FB_FEATS[GAMMAS[0]],
        kind_tr=mcbm_kind_tr, ids_tr=mcbm_ids_tr,
        kind_te=mcbm_kind_te, ids_te=mcbm_ids_te,
        layer=LAYER,
        min_pos_per_species=2, min_species_with_pos=2,
        min_overall_prev=0.001, max_overall_prev=0.999,
        epochs=6, verbose_every=5,
    )
    print(f'Retry: {len(screen_df)} concepts passed')

SCREENED_ATTR_LIST = screen_df['attr'].tolist() if len(screen_df) > 0 else ATTR_LIST
if screen_df.empty:
    print('[warn] No concepts passed even with loose filters — using all 26.')
else:
    print('SCREENED_ATTR_LIST:', SCREENED_ATTR_LIST)
    display(screen_df)

attr_to_spread = screen_df.set_index('attr')['recall_p90_p10'].to_dict() if not screen_df.empty else {}


In [ ]:
def run_many(
    attr_list, model_name, feat_dir,
    kind_tr, ids_tr, kind_te, ids_te,
    *, layer, min_certainty=1, thr=0.5, epochs=25,
    min_each=10, n_pairs=200, B_gap=100, B_species=100,
    use_balanced_acc=None,
):
    """
    Runs run_one_attribute over a list of attrs.
    Loads features once per layer (not once per attribute) for efficiency.
    Returns: info_df, pairs_df, species_df.
    """
    Xtr_all = load_features(feat_dir, layer, 'train')
    Xte_all = load_features(feat_dir, layer, 'test')

    all_info, all_pair_summ, all_species = [], [], []
    for attr in attr_list:
        info, _, pair_summ, _, species_table = run_one_attribute(
            attr, feat_dir, kind_tr, ids_tr, kind_te, ids_te,
            layer=layer, Xtr_all=Xtr_all, Xte_all=Xte_all,
            min_certainty=min_certainty, thr=thr, epochs=epochs,
            min_each=min_each, n_pairs=n_pairs, B_gap=B_gap, B_species=B_species,
            use_balanced_acc=use_balanced_acc,
        )
        info = dict(info); info['model'] = model_name
        all_info.append(info)
        if pair_summ is not None and len(pair_summ):
            ps = pair_summ.copy(); ps['attr'] = attr; ps['model'] = model_name
            all_pair_summ.append(ps)
        st = species_table.copy(); st['attr'] = attr; st['model'] = model_name
        all_species.append(st)
        acc_label = 'bal_acc' if (EMERGE_METRIC == 'balanced') else 'test_acc'
        print(model_name, attr, f'{acc_label}=', round(info['test_acc'], 4),
              'mean_gap=', round(info['mean_gap'], 4) if not np.isnan(info['mean_gap']) else 'nan')

    info_df    = pd.DataFrame(all_info)
    pairs_df   = pd.concat(all_pair_summ, ignore_index=True) if all_pair_summ else pd.DataFrame()
    species_df = pd.concat(all_species,   ignore_index=True) if all_species   else pd.DataFrame()
    return info_df, pairs_df, species_df


print('Defined: run_many')

## 5. Recall gap analysis — γ sweep

`run_many` runs the full **matched-pair recall gap** pipeline for each γ value.

**FunnyBirds pairing strategy:** concepts are 100% present or 100% absent per species,
so we pair all-positive species and measure whether the probe fires equally well for both.
A large gap means the probe has learned species identity, not just the concept.

**Central prediction:** as γ increases (stronger IB penalty), the MCBM backbone
retains less species-identity information → recall gap should *decrease*.

In [ ]:
mcbm_results = {}

for g in GAMMAS:
    print(f"\n{'='*60}")
    print(f"  Running MCBM gamma={g}")
    print(f"{'='*60}")
    info, pairs, species = run_many(
        SCREENED_ATTR_LIST, f'mcbm_fb_gamma{g}', MCBM_FB_FEATS[g],
        mcbm_kind_tr, mcbm_ids_tr, mcbm_kind_te, mcbm_ids_te,
        layer=LAYER, thr=0.5, n_pairs=200, B_gap=100, B_species=100,
    )
    mcbm_results[g] = (info, pairs, species)

print("\nDone. Results in mcbm_results[gamma].")

In [ ]:
for g in GAMMAS:
    _, _, species = mcbm_results[g]
    out_csv = f'fb_mcbm_species_gamma{g}.csv'
    species.to_csv(out_csv, index=False)
    print(f'Saved {out_csv}  ({len(species)} rows)')

In [ ]:
rows = []
for g in GAMMAS:
    info, _, _ = mcbm_results[g]
    rows.append({
        'gamma': g,
        'n_concepts': len(info),
        'mean_gap': float(info['mean_gap'].mean()),
        'median_gap': float(info['mean_gap'].median()),
        'frac_gap_gt0': float((info['mean_gap'] > 0).mean()),
        'mean_probe_acc': float(info['test_acc'].mean()),
    })

summary_table = pd.DataFrame(rows)
print('Summary across gammas:')
display(summary_table)

In [ ]:
# Gamma=0 (CBM-equivalent) vs max gamma — mirrors baseline vs CBM table
# in funnybirds_cbm_recall.ipynb cell 32.
g_lo, g_hi = GAMMAS[0], GAMMAS[-1]
info_lo, _, _ = mcbm_results[g_lo]
info_hi, _, _ = mcbm_results[g_hi]

delta_t = (
    info_lo[['attr','test_acc','mean_gap']]
    .rename(columns={'test_acc': 'acc_lo', 'mean_gap': 'gap_lo'})
    .merge(
        info_hi[['attr','test_acc','mean_gap']]
        .rename(columns={'test_acc': 'acc_hi', 'mean_gap': 'gap_hi'}),
        on='attr', how='outer')
)
delta_t['delta_gap'] = delta_t['gap_hi'] - delta_t['gap_lo']  # negative = IB helped
delta_t['delta_acc'] = delta_t['acc_hi'] - delta_t['acc_lo']
delta_t = delta_t.sort_values('gap_lo', ascending=False)
print(f'gamma={g_lo} vs gamma={g_hi}  (negative delta_gap = IB reduced entanglement)')
display(delta_t)
delta_t.to_csv(f'fb_mcbm_delta_g{g_lo}_vs_g{g_hi}.csv', index=False)
print(f'Saved fb_mcbm_delta_g{{g_lo}}_vs_g{{g_hi}}.csv')


In [ ]:
# Scatter: gamma=0 vs max_gamma — mirrors funnybirds_cbm_recall.ipynb cell 33.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(delta_t['gap_lo'], delta_t['gap_hi'], alpha=0.75, s=60)
for _, row in delta_t.dropna(subset=['gap_lo','gap_hi']).iterrows():
    ax.annotate(row['attr'], (row['gap_lo'], row['gap_hi']), fontsize=6, alpha=0.6)
lim = max(delta_t[['gap_lo','gap_hi']].max().max(), 0.05) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='y=x (no change)')
ax.set_xlabel(f'Recall gap  gamma={g_lo} (CBM-equivalent)')
ax.set_ylabel(f'Recall gap  gamma={g_hi} (max IB)')
ax.set_title(f'Recall gap: gamma={g_lo} vs gamma={g_hi}\n(below diagonal = IB reduced entanglement)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(delta_t['acc_lo'], delta_t['acc_hi'], alpha=0.75, s=60)
for _, row in delta_t.dropna(subset=['acc_lo','acc_hi']).iterrows():
    ax.annotate(row['attr'], (row['acc_lo'], row['acc_hi']), fontsize=6, alpha=0.6)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='y=x')
ax.set_xlabel(f'Probe accuracy  gamma={g_lo}')
ax.set_ylabel(f'Probe accuracy  gamma={g_hi}')
ax.set_title(f'Probe accuracy: gamma={g_lo} vs gamma={g_hi}\n(below diagonal = IB degraded concept decodability)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'FunnyBirds MCBM: gamma={g_lo} vs gamma={g_hi}', y=1.02)
plt.tight_layout()
plt.savefig(f'fb_mcbm_scatter_g{{g_lo}}_vs_g{{g_hi}}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved fb_mcbm_scatter_g{{g_lo}}_vs_g{{g_hi}}.png')


In [ ]:
# Species recall sanity check at gamma=0 (should be close to cbm_fb results).
# Mirrors mcbm_recall_full.ipynb cell 32.
_, _, species_ref = mcbm_results[GAMMAS[0]]
print(f'MCBM gamma={GAMMAS[0]} — per-species recall sorted by true positives:')
species_ref.sort_values('tp', ascending=False).head(30)


### Interpreting matched-pair gap columns

Each row corresponds to a *species pair* evaluated for one attribute and γ value.

| Column | Meaning |
|---|---|
| **gap_mean** | Mean absolute recall difference between the two species across bootstrap runs |
| **gap_std** | Standard deviation of the gap |
| **gap_snr** | `gap_mean / gap_std` — higher = more consistently observed |
| **gap_ci_lo / gap_ci_hi** | 95% percentile bootstrap CI. Excludes 0 → gap stable under resampling |
| **gap_p** | Two-sided bootstrap p-value for H₀: gap = 0 |
| **npos** | Positive images sampled per species per bootstrap run |
| **n_runs** | Bootstrap iterations used to estimate the gap |

**FunnyBirds note:** `nneg = 0` always — we pair all-positive species (no within-species negatives).
The gap is purely a cross-species recall difference for species that both truly have the concept.

## 5b. Per-attribute summary tables

For each γ value: per-attribute aggregation of matched-pair recall gaps.

| Column | Meaning |
|---|---|
| **gap_mean** | Mean abs recall gap across all sampled species pairs |
| **gap_max** | Worst single pair gap |
| **frac_p_small** | Fraction of pairs with bootstrap p ≤ 0.05 |
| **frac_ci_above0** | Fraction of pairs whose 95% CI lower bound > 0 |
| **gap_snr_mean** | Mean signal-to-noise ratio across pairs |

In [ ]:
def summarize_by_attr(pairs_df: pd.DataFrame):
    if pairs_df.empty:
        return pairs_df
    g = pairs_df.groupby(['model','attr'], as_index=False)
    return g.agg(
        gap_mean=('gap_mean','mean'),
        gap_median=('gap_mean','median'),
        gap_max=('gap_mean','max'),
        n_pairs=('gap_mean','size'),
        frac_p_small=('gap_p', lambda s: float(np.mean(np.asarray(s) <= 0.05))),
        frac_ci_above0=('gap_ci_lo', lambda s: float(np.mean(np.asarray(s) > 0))),
        gap_snr_mean=('gap_snr','mean'),
    ).sort_values(['model','gap_mean'], ascending=[True, False])


def add_gap_interpretability_cols(pair_df: pd.DataFrame) -> pd.DataFrame:
    out = pair_df.copy()
    out['gap_snr']  = out['gap_mean'] / out['gap_std'].replace(0, np.nan)
    out['prev_matched'] = (out['npos'] / (out['npos'] + out['nneg'])).astype(float)
    out['gap_norm'] = out['gap_mean'] / 1.0
    return out


for g in GAMMAS:
    _, pairs, _ = mcbm_results[g]
    if not pairs.empty:
        s = summarize_by_attr(add_gap_interpretability_cols(pairs))
        print(f"\n{'='*55}")
        print(f'  Per-attribute summary   gamma = {g}')
        print(f"{'='*55}")
        display(s)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Panel A: mean recall gap vs gamma, per attribute
ax = axes[0]
for attr in SCREENED_ATTR_LIST:
    gaps = []
    for g in GAMMAS:
        info, _, _ = mcbm_results[g]
        row = info[info['attr'] == attr]
        gaps.append(float(row['mean_gap'].iloc[0]) if len(row) > 0 else np.nan)
    ax.plot(GAMMAS, gaps, marker='o', label=attr, alpha=0.7)
ax.set_xlabel('gamma (IB penalty strength)')
ax.set_ylabel('Mean matched-pair recall gap')
ax.set_title('FunnyBirds MCBM: Recall gap vs gamma\n(hypothesis: decreases as gamma increases)')
ax.legend(fontsize=6, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)

# Panel B: mean gap averaged over all concepts (with SEM shading)
ax = axes[1]
all_means = []
for g in GAMMAS:
    info, _, _ = mcbm_results[g]
    all_means.append(info['mean_gap'].dropna().values)
means_arr = [np.mean(v) for v in all_means]
sems_arr  = [np.std(v) / np.sqrt(len(v)) for v in all_means]
ax.errorbar(GAMMAS, means_arr, yerr=sems_arr, marker='o', capsize=4, color='steelblue')
ax.fill_between(GAMMAS,
                [m - s for m, s in zip(means_arr, sems_arr)],
                [m + s for m, s in zip(means_arr, sems_arr)],
                alpha=0.2, color='steelblue')
ax.set_xlabel('gamma (IB penalty strength)')
ax.set_ylabel('Mean gap ± SEM (across concepts)')
ax.set_title('Mean recall gap across all concepts vs gamma')
ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds MCBM: Recall gap vs gamma', y=1.02)
plt.tight_layout()
plt.savefig('fb_mcbm_recall_gap_vs_gamma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_mcbm_recall_gap_vs_gamma.png')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for attr in SCREENED_ATTR_LIST:
    accs = []
    for g in GAMMAS:
        info, _, _ = mcbm_results[g]
        row = info[info['attr'] == attr]
        accs.append(float(row['test_acc'].iloc[0]) if len(row) > 0 else np.nan)
    ax.plot(GAMMAS, accs, marker='s', label=attr, alpha=0.7)
ax.set_xlabel('gamma (IB penalty strength)')
ax.set_ylabel('Test accuracy (attribute probe)')
ax.set_title('FunnyBirds MCBM: Probe accuracy vs gamma\n(drop = IB suppresses concept information)')
ax.legend(fontsize=6, loc='lower left', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fb_mcbm_testacc_vs_gamma.png', dpi=150)
plt.show()
print('Saved fb_mcbm_testacc_vs_gamma.png')

## 5c. FunnyBirds-specific: Concept conformance by gamma

For each concept, the **class-concept matrix** tells us exactly which species should
have the concept (all-positive).  We can therefore measure how consistently the probe
fires across these GT-positive species — the **recall range** among GT-positive species.

A lower recall range at higher gamma = IB reducing species-identity leakage.

In [ ]:
def concept_conformance_by_gamma(attr_name, cc_df, mcbm_results, gammas, species_id_offset=0):
    """
    For a concept, use the exact class-concept matrix to identify GT-positive species.
    For each gamma, extract recall of each GT-positive species and report:
      - recall_mean  : mean recall over GT-positive species
      - recall_range : max - min recall (entanglement measure)
      - recall_std   : std of recall
    Lower recall_range at higher gamma = less species-identity leakage.
    """
    # GT-positive species from class-concept matrix (row index = species_id - offset)
    gt_positive_rows = cc_df.index[cc_df[attr_name] == 1].tolist()
    # Convert row names to species_ids (funnybird_00 -> 0)
    gt_positive_sids = [int(r.split('_')[-1]) + species_id_offset for r in gt_positive_rows]

    rows = []
    for g in gammas:
        _, _, species_df = mcbm_results[g]
        sub = species_df[
            (species_df['attr'] == attr_name) &
            (species_df['species_id'].isin(gt_positive_sids))
        ]
        recalls = sub['recall'].dropna().values
        rows.append({
            'gamma': g,
            'n_gt_pos_species': len(gt_positive_sids),
            'n_with_recall': len(recalls),
            'recall_mean':  float(np.mean(recalls))  if len(recalls) > 0 else np.nan,
            'recall_range': float(np.ptp(recalls))   if len(recalls) > 1 else np.nan,
            'recall_std':   float(np.std(recalls))   if len(recalls) > 1 else np.nan,
        })
    return pd.DataFrame(rows)


# Show conformance for each screened concept
for attr in SCREENED_ATTR_LIST:
    conf = concept_conformance_by_gamma(attr, cc_df, mcbm_results, GAMMAS, species_id_offset=_sid_min)
    print(f'\n--- {attr} ---')
    display(conf)

In [ ]:
# Line plot: recall_range vs gamma per concept.
# recall_range = max - min recall over GT-positive species.
# Lower at higher gamma = IB suppressing species-identity leakage.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

conf_data = {}
for attr in SCREENED_ATTR_LIST:
    conf_data[attr] = concept_conformance_by_gamma(
        attr, cc_df, mcbm_results, GAMMAS, species_id_offset=_sid_min
    )

ax = axes[0]
for attr, conf in conf_data.items():
    ax.plot(conf['gamma'], conf['recall_range'], marker='o', label=attr, alpha=0.7)
ax.set_xlabel('gamma')
ax.set_ylabel('Recall range (max − min over GT-positive species)')
ax.set_title('Concept conformance: recall_range vs gamma\n(lower = IB reducing species-leakage)')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

ax = axes[1]
for attr, conf in conf_data.items():
    ax.plot(conf['gamma'], conf['recall_mean'], marker='s', label=attr, alpha=0.7)
ax.set_xlabel('gamma')
ax.set_ylabel('Mean recall over GT-positive species')
ax.set_title('Mean recall (GT-positive species) vs gamma\n(flat = concept preserved; drop = IB too strong)')
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds MCBM: Concept conformance by gamma (exact GT from class-concept matrix)', y=1.02)
plt.tight_layout()
plt.savefig('fb_mcbm_concept_conformance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_mcbm_concept_conformance.png')


## 5d. FunnyBirds-specific: Part-group analysis

FunnyBirds concepts are organised by body part (beak / wing / tail / foot / eye).
We test whether the backwash effect (gap reduction with gamma) differs across parts:
- **Tail** (9 variants): most species-unique shapes → expect high entanglement
- **Eye** (3 variants): shared shapes → expect lower entanglement

In [ ]:
PART_GROUPS = {
    part: [a for a in SCREENED_ATTR_LIST if a.startswith(f'{part}_')]
    for part in ['beak', 'wing', 'tail', 'foot', 'eye']
}
print('Part groups (screened concepts):')
for part, attrs in PART_GROUPS.items():
    print(f'  {part}: {attrs}')

# Plot mean recall gap per part group vs gamma
fig, ax = plt.subplots(figsize=(9, 4))
PART_COLORS = {'beak': 'steelblue', 'wing': 'seagreen', 'tail': 'crimson',
               'foot': 'darkorange', 'eye': 'purple'}

for part, attrs in PART_GROUPS.items():
    if not attrs:
        continue
    means, sems = [], []
    for g in GAMMAS:
        info, _, _ = mcbm_results[g]
        sub = info[info['attr'].isin(attrs)]['mean_gap'].dropna().values
        means.append(np.mean(sub) if len(sub) > 0 else np.nan)
        sems.append(np.std(sub) / np.sqrt(len(sub)) if len(sub) > 1 else 0)
    ax.errorbar(GAMMAS, means, yerr=sems, marker='o', label=part,
                color=PART_COLORS.get(part), capsize=3)

ax.set_xlabel('gamma (IB penalty strength)')
ax.set_ylabel('Mean recall gap (mean ± SEM across concepts)')
ax.set_title('FunnyBirds MCBM: Recall gap vs gamma by body part')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fb_mcbm_gap_by_part.png', dpi=150)
plt.show()
print('Saved fb_mcbm_gap_by_part.png')

## 5e. FunnyBirds-specific: Concept overlap vs recall gap

Two species that share many concept variants (similar overall appearance) may show
larger recall gaps because the probe is confused by their visual similarity.
We test this using the exact class-concept matrix.

In [ ]:
def species_concept_overlap(cc_df, sid_A, sid_B, species_id_offset=0):
    """
    Number of concept variants shared between species A and B.
    Overlap = sum over concepts of (cc[A, c] AND cc[B, c]).
    """
    row_A = f'funnybird_{int(sid_A) - species_id_offset:02d}'
    row_B = f'funnybird_{int(sid_B) - species_id_offset:02d}'
    if row_A not in cc_df.index or row_B not in cc_df.index:
        return np.nan
    return int((cc_df.loc[row_A] & cc_df.loc[row_B]).astype(bool).sum())


def annotate_pairs_with_concept_overlap(pairs_df: pd.DataFrame, cc_df, species_id_offset=0):
    """
    Add n_shared_concepts and overlap_ratio to a pairs_df.
    overlap_ratio = n_shared / total_concepts (0..1).
    """
    df = pairs_df.copy()
    n_total = len(cc_df.columns)
    df['n_shared_concepts'] = df.apply(
        lambda r: species_concept_overlap(cc_df, r['sid_A'], r['sid_B'], species_id_offset),
        axis=1,
    )
    df['overlap_ratio'] = df['n_shared_concepts'] / n_total
    return df


# Annotate pairs for each gamma and plot overlap vs gap
fig, axes = plt.subplots(1, len(GAMMAS), figsize=(4 * len(GAMMAS), 4), sharey=True)
if len(GAMMAS) == 1:
    axes = [axes]

for ax, g in zip(axes, GAMMAS):
    _, pairs, _ = mcbm_results[g]
    if pairs.empty:
        ax.set_title(f'gamma={g}\n(no pairs)')
        continue
    pairs_ann = annotate_pairs_with_concept_overlap(pairs, cc_df, species_id_offset=_sid_min)
    ax.scatter(pairs_ann['n_shared_concepts'], pairs_ann['gap_mean'], alpha=0.3, s=12)
    # Trend line
    v = pairs_ann[['n_shared_concepts','gap_mean']].dropna()
    if len(v) > 2:
        m, b = np.polyfit(v['n_shared_concepts'], v['gap_mean'], 1)
        xs = np.array([v['n_shared_concepts'].min(), v['n_shared_concepts'].max()])
        ax.plot(xs, m * xs + b, 'r--', alpha=0.7, label=f'slope={m:.3f}')
        ax.legend(fontsize=8)
    ax.set_xlabel('Shared concept variants')
    ax.set_title(f'gamma={g}')
    if g == GAMMAS[0]:
        ax.set_ylabel('Mean recall gap')
    ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds MCBM: Concept overlap vs recall gap (by gamma)', y=1.02)
plt.tight_layout()
plt.savefig('fb_mcbm_overlap_vs_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_mcbm_overlap_vs_gap.png')

## 5f. Layer sweep, attribute emergence, and entanglement signal

**Why sweep all 18 layers?**
The pre/at/post grouping requires knowing *when* each attribute becomes linearly decodable
(its emergence layer). We sweep all layers, train a probe at each, and record the
accuracy curve.

**Emergence criteria** (controlled by `EMERGE_CRITERION` above):
- `sharp_jump` — index of the largest single-step increase in the accuracy curve
- `frac90` — first layer index reaching `EMERGE_FRAC` × final accuracy

**Species emergence layer:** `layer4.0` = index 14 in LAYERS (from `lfcbm_newrecall.ipynb`).

**Pre / At / Post grouping:**
- **pre**  emergence_idx < 14: attribute decodable without species-level cues
- **at**   emergence_idx = 14: attribute and species emerge together
- **post** emergence_idx > 14: attribute only decodable after species representations form

**Central prediction:** POST attributes should show larger recall gaps and larger gamma-
driven gap reduction (their evidence is entangled with species-level representations).

In [ ]:
LAYER_TO_IDX = {l: i for i, l in enumerate(LAYERS)}


def sharp_jump_idx(vals):
    """Index of the largest single-step increase in a layerwise accuracy curve."""
    v = np.asarray(vals, dtype=float)
    return int(np.argmax(np.diff(v))) + 1


def frac_of_final_idx(vals, frac=0.90):
    """First layer index where the curve reaches `frac` × its final value."""
    v = np.asarray(vals, dtype=float)
    target = frac * v[-1]
    idxs = np.where(v >= target)[0]
    return int(idxs[0]) if len(idxs) else len(v) - 1


def compute_emergence_for_attr(
    attr_name: str,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    *,
    use_balanced_acc: bool = True,
    epochs: int = 6,
):
    """
    Trains a probe at each of the 18 LAYERS; returns layerwise accuracy curve.
    Returns: curve (list[float]), emerge_idx_jump (int), emerge_idx_frac (int).
    """
    curve = []
    for layer in LAYERS:
        info, _, _, _, _ = run_one_attribute(
            attr_name, feat_dir, kind_tr, ids_tr, kind_te, ids_te,
            layer=layer, epochs=epochs, n_pairs=0, B_species=0, B_gap=0,
            use_balanced_acc=use_balanced_acc,
        )
        curve.append(info['test_acc'])

    sj  = sharp_jump_idx(curve)
    f90 = frac_of_final_idx(curve, frac=EMERGE_FRAC)
    return curve, sj, f90


def screen_and_compute_emergence(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    *,
    use_balanced_acc: bool = True,
    epochs: int = 6,
    max_attrs: int = 26,
    verbose_every: int = 5,
) -> pd.DataFrame:
    rows  = []
    cands = list(candidate_attrs)[:max_attrs]
    for i, attr in enumerate(cands):
        try:
            curve, sj, f90 = compute_emergence_for_attr(
                attr, feat_dir, kind_tr, ids_tr, kind_te, ids_te,
                use_balanced_acc=use_balanced_acc, epochs=epochs,
            )
            rows.append({
                'attr': attr,
                'emerge_idx_jump': sj, 'emerge_layer_jump': LAYERS[sj],
                'emerge_idx_frac': f90, 'emerge_layer_frac': LAYERS[f90],
                'final_acc': float(curve[-1]),
                'curve': curve,
            })
        except Exception:
            pass
        if verbose_every and (i + 1) % verbose_every == 0:
            print(f'  [{i+1}/{len(cands)}] done  (kept: {len(rows)})')

    df = pd.DataFrame(rows)
    print(f'\nEmergence sweep: {len(df)}/{len(cands)} attrs kept.')
    return df


print('Defined: sharp_jump_idx  frac_of_final_idx  compute_emergence_for_attr  screen_and_compute_emergence')

In [ ]:
# Run emergence sweep using GAMMAS[0] features (most information-rich)
attr_emerge_df = screen_and_compute_emergence(
    SCREENED_ATTR_LIST,
    MCBM_FB_FEATS[GAMMAS[0]],
    mcbm_kind_tr, mcbm_ids_tr,
    mcbm_kind_te, mcbm_ids_te,
    use_balanced_acc=(EMERGE_METRIC == 'balanced'),
    epochs=6,
    max_attrs=len(SCREENED_ATTR_LIST),
    verbose_every=5,
)

# Classify pre / at / post relative to species emergence layer
attr_emerge_df['group'] = attr_emerge_df[emerge_col].apply(
    lambda x: 'pre' if x < SPECIES_LAYER_IDX else ('at' if x == SPECIES_LAYER_IDX else 'post')
)

print(f'Emergence criterion: {EMERGE_CRITERION}  (column: {emerge_col})')
print('Group counts:', dict(attr_emerge_df['group'].value_counts()))
display(attr_emerge_df[['attr', emerge_col, layer_col, 'final_acc', 'group']])

In [ ]:
# Plot layerwise accuracy curves coloured by pre/at/post group
GROUP_COLORS = {'pre': 'steelblue', 'at': 'orange', 'post': 'crimson'}

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(LAYERS))
for _, row in attr_emerge_df.iterrows():
    color = GROUP_COLORS.get(row['group'], 'gray')
    ax.plot(x, row['curve'], alpha=0.6, color=color, label=row['group'])
    ax.text(x[-1] + 0.1, row['curve'][-1], row['attr'], fontsize=6, color=color)

ax.axvline(SPECIES_LAYER_IDX, color='black', ls='--', alpha=0.7, label='species layer (layer4.0)')

# Legend: one entry per group
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], color=c, label=g) for g, c in GROUP_COLORS.items()]
handles.append(Line2D([0],[0], color='black', ls='--', label='species layer'))
ax.legend(handles=handles, fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(LAYERS, rotation=45, ha='right', fontsize=7)
ax.set_ylabel(f'Probe accuracy ({EMERGE_METRIC})')
ax.set_title(f'FunnyBirds MCBM (gamma={GAMMAS[0]}): Attribute emergence curves')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fb_mcbm_emergence_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_mcbm_emergence_curves.png')

In [ ]:
# Build per-gamma group-level recall gap summary
GROUP_ORDER = ['pre', 'at', 'post']
group_gamma_rows = []

for g in GAMMAS:
    info_g, pairs_g, _ = mcbm_results[g]
    grp = info_g.merge(attr_emerge_df[['attr','group']], on='attr', how='left')
    for gname in GROUP_ORDER:
        sub = grp[grp['group'] == gname]['mean_gap'].dropna()
        n   = len(sub)
        # frac_ci_above0: fraction of pairs in this group whose CI lower bound > 0
        if not pairs_g.empty:
            attrs_in_group = grp[grp['group'] == gname]['attr'].tolist()
            pg = pairs_g[pairs_g['attr'].isin(attrs_in_group)]
            frac_disc = float((np.asarray(pg['gap_ci_lo']) > 0).mean()) if len(pg) > 0 else np.nan
        else:
            frac_disc = np.nan
        group_gamma_rows.append({
            'gamma': g, 'group': gname, 'n': n,
            'mean_gap': float(sub.mean()) if n > 0 else np.nan,
            'sem_gap': float(sub.std() / np.sqrt(n)) if n > 1 else np.nan,
            'frac_discriminative': frac_disc,
        })

group_gamma_df = pd.DataFrame(group_gamma_rows)
print('Group-gamma summary:')
display(group_gamma_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel A: recall gap vs gamma per pre/at/post group
ax = axes[0]
for gname in GROUP_ORDER:
    sub = group_gamma_df[group_gamma_df['group'] == gname]
    ax.errorbar(sub['gamma'], sub['mean_gap'], yerr=sub['sem_gap'].fillna(0),
                marker='o', label=gname, color=GROUP_COLORS[gname], capsize=3)
ax.set_xlabel('gamma (IB penalty strength)')
ax.set_ylabel('Mean recall gap (± SEM)')
ax.set_title('Does IB reduce entanglement?\n(hypothesis: post line slopes down)')
ax.legend(); ax.grid(True, alpha=0.3)

# Panel B: fraction discriminative vs gamma per group
ax = axes[1]
for gname in GROUP_ORDER:
    sub = group_gamma_df[group_gamma_df['group'] == gname]
    ax.plot(sub['gamma'], sub['frac_discriminative'], marker='s',
            label=gname, color=GROUP_COLORS[gname])
ax.axhline(0.5, color='gray', ls='--', alpha=0.6, label='50% line')
ax.set_xlabel('gamma')
ax.set_ylabel('Fraction discriminative (CI lo > 0)')
ax.set_title('Fraction discriminative vs gamma')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# Panel C: emergence index vs mean gap (at gamma=GAMMAS[0])
ax = axes[2]
info_g0, _, _ = mcbm_results[GAMMAS[0]]
gap_info = info_g0.merge(attr_emerge_df[['attr', emerge_col, 'group']], on='attr', how='left')
for gname in GROUP_ORDER:
    sub = gap_info[gap_info['group'] == gname]
    ax.scatter(sub[emerge_col], sub['mean_gap'], color=GROUP_COLORS[gname],
               alpha=0.7, s=50, label=f'{gname} (n={len(sub)})')
v2 = gap_info[[emerge_col, 'mean_gap']].dropna()
if len(v2) > 2:
    m, b = np.polyfit(v2[emerge_col], v2['mean_gap'], 1)
    xs = np.array([v2[emerge_col].min(), v2[emerge_col].max()])
    ax.plot(xs, m * xs + b, 'k--', label=f'trend (slope={m:.4f})')
ax.axvline(SPECIES_LAYER_IDX, color='gray', ls=':', label='species layer')
ax.set_xlabel(f'Concept emergence layer ({EMERGE_CRITERION})')
ax.set_ylabel('Mean recall gap (gamma=GAMMAS[0])')
ax.set_title('Later-emerging -> larger gap?\n(continuous, no binning)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(
    f'FunnyBirds MCBM: Entanglement signal (criterion: {EMERGE_CRITERION}, metric: {EMERGE_METRIC})',
    y=1.02,
)
plt.tight_layout()
plt.savefig('fb_mcbm_entanglement_signal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_mcbm_entanglement_signal.png')

### How confidence intervals and p-values are computed

For each species pair, `fb_bootstrap_summary` runs `fb_pair_eval` B times
(different seeds → different bootstrap samples with replacement).

**Bootstrap CI (95%):** `gap_ci_lo` = 2.5th percentile, `gap_ci_hi` = 97.5th percentile
across B gap values. CI excludes 0 → gap is stable under resampling.

**Bootstrap p-value:** `p = 2 × min(P(gap ≤ 0), P(gap ≥ 0))`. Significant if p ≤ 0.05
*and* CI excludes 0.

**FunnyBirds:** `replace=True` sampling ensures different seeds give genuinely different
subsets (no negatives to permute). The distribution reflects how consistently the probe
separates two all-positive species.

## 5g. Top species pairs per attribute and gamma

In [ ]:
def top_pairs(pairs_df: pd.DataFrame, model: str, attr: str, k=10):
    sub  = pairs_df[(pairs_df['model'] == model) & (pairs_df['attr'] == attr)].copy()
    if sub.empty:
        return sub
    cols = ['species_A','species_B','gap_mean','gap_ci_lo','gap_ci_hi',
            'gap_p','gap_std','gap_snr','npos','n_runs']
    cols = [c for c in cols if c in sub.columns]
    return sub.sort_values('gap_mean', ascending=False).head(k)[cols]


for g in GAMMAS:
    _, pairs, _ = mcbm_results[g]
    if pairs.empty:
        print(f'gamma={g}: no pairs data')
        continue
    pairs_enr = add_gap_interpretability_cols(pairs)
    print(f"\n{'='*60}")
    print(f'  Top species pairs   gamma = {g}')
    print(f"{'='*60}")
    for attr in SCREENED_ATTR_LIST:
        print(f'\nAttribute: {attr}')
        display(top_pairs(pairs_enr, f'mcbm_fb_gamma{g}', attr, k=5))

## 6. Cross-model comparison table

Aggregates per-species recall tables from:

| Model | File | Notes |
|---|---|---|
| `baseline` | `fb_baseline_species.csv` | Plain ResNet-50, no concept supervision |
| `cbm` | `fb_cbm_species.csv` | SimpleCBM, no IB |
| `mcbm_γ=X` | `fb_mcbm_species_gamma{X}.csv` | MCBM, various γ (this notebook) |

In [ ]:
dfs = []
for name, path in [
    ('baseline', 'fb_baseline_species.csv'),
    ('cbm',      'fb_cbm_species.csv'),
]:
    p = Path(path)
    if p.exists():
        dfs.append(pd.read_csv(p).assign(model=name))
    else:
        print(f'[warn] {path} not found — run funnybirds_cbm_recall.ipynb first.')

for g in GAMMAS:
    p = Path(f'fb_mcbm_species_gamma{g}.csv')
    if p.exists():
        dfs.append(pd.read_csv(p).assign(model=f'mcbm_gamma{g}'))

if dfs:
    all_df = pd.concat(dfs, ignore_index=True)
    if 'attr' in all_df.columns and 'recall' in all_df.columns:
        summary = (
            all_df.groupby(['model','attr'])['recall']
            .mean().reset_index().rename(columns={'recall': 'mean_recall'})
            .sort_values(['attr','model'])
        )
        display(summary)
    else:
        display(all_df.head())
else:
    print('No species CSV files found. Run the analysis cells above first.')